# Drift y reentrenamiento de Green Taxi

Completa la preparación de la guía y abre este notebook desde la raíz del repositorio del equipo. Selecciona el intérprete de `.venv`. El análisis compara marzo con abril. Cuando se activa el entrenamiento, ambos modelos se evalúan con mayo.

## Ejecutar los tres escenarios

El notebook utiliza los Parquet si encuentra los tres archivos. En otro caso, utiliza los tres CSV preparados en `datos/`. La primera celda muestra qué fuente va a usar. Prefect ejecuta el flujo en un servidor temporal local.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

raiz = Path.cwd()
meses = ("03", "04", "05")
parquet = [raiz / "datos" / f"green_tripdata_2026-{mes}.parquet" for mes in meses]
muestras = [raiz / "datos" / f"green-taxi-2026-{mes}.csv" for mes in meses]
if not (raiz / "drift.py").is_file():
    raise FileNotFoundError("Abre el notebook desde la raíz del repositorio del laboratorio.")
if all(ruta.is_file() for ruta in parquet):
    opcion_fuente = []
elif all(ruta.is_file() for ruta in muestras):
    opcion_fuente = ["--muestras"]
else:
    raise FileNotFoundError("Prepara los tres Parquet o los tres CSV indicados en la guía.")

entorno = os.environ.copy()
entorno.pop("PREFECT_API_URL", None)
entorno.pop("PREFECT_API_KEY", None)
entorno.update(
    PREFECT_HOME=str(raiz / ".prefect"),
    PREFECT_SERVER_EPHEMERAL_ENABLED="true",
    PREFECT_SERVER_ANALYTICS_ENABLED="false",
)
print("Fuente:", "CSV de contingencia" if opcion_fuente else "Parquet de TLC")

In [ ]:
for escenario in ("control", "historico", "controlado"):
    subprocess.run(
        [sys.executable, "drift.py", escenario, *opcion_fuente],
        check=True,
        env=entorno,
    )

In [ ]:
for escenario in ("control", "historico", "controlado"):
    ruta = raiz / "resultados" / f"drift-{escenario}.json"
    resultado = json.loads(ruta.read_text(encoding="utf-8"))
    print(escenario, round(resultado["cambio_pp"], 2), "pp", resultado["decision"])

## Interpretar

Compara el umbral con el cambio observado. Identifica qué escenario inicia el entrenamiento y qué diferencia de RMSE llevó a aceptar o conservar el modelo. Usa los JSON y la salida visible como evidencia.